## Wav2Vec2 model with HuggingFace

In [6]:
import torch
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
from transformers import Wav2Vec2ForSequenceClassification

In [ ]:
def export_wav2vec2_to_onnx(
    model_name: str = "facebook/wav2vec2-base",
    onnx_path: str = "wav2vec2-base.onnx",
    opset_version: int = 12,
    seq_length: int = 16000,
):
    """
    Loads Wav2Vec2Model and Wav2Vec2FeatureExtractor, prepares a dummy input,
    and exports the model to ONNX format.
    """
    # 1) Load the feature extractor and model
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
    model = Wav2Vec2Model.from_pretrained(model_name)
    model.eval()

    # 2) Create a dummy input (1 second of audio at 16 kHz)
    #    You can adjust seq_length for different durations.
    #    The feature extractor normalizes/clamps internally,
    #    but for export, it's enough to pass floats in [-1, 1].
    dummy_audio = torch.randn(1, seq_length, dtype=torch.float32)

    # 3) Pass through the extractor to get input_values
    inputs = feature_extractor(
        dummy_audio.numpy(),
        sampling_rate=feature_extractor.sampling_rate,
        return_tensors="pt",
    )

    input_values = inputs.input_values  # Tensor [1, seq_length]

    # 4) Define names and dynamic axes (for variable-length input)
    input_names  = ["input_values"]
    output_names = ["last_hidden_state"]
    dynamic_axes = {
        "input_values": {1: "seq_len"},         # axis 1 = sequence length
        "last_hidden_state": {1: "seq_len"},    # same length in output
    }

    # 5) Export to ONNX
    torch.onnx.export(
        model,                               # model
        (input_values,),                     # input tuple
        onnx_path,                           # output path
        opset_version=opset_version,
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        do_constant_folding=True,            # optimize constants
        verbose=True,
    )
    print(f"✅ Model successfully exported to {onnx_path}")


## Load the model with Optimum[HuggingFace]

### Load the model and save it in ONNX format

In [ ]:
from transformers import Wav2Vec2FeatureExtractor
from optimum.onnxruntime import ORTModelForAudioClassification

model_name = "facebook/wav2vec2-base"

# 1) Download the feature extractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

# 2) Export to ONNX using audio classification
ort_model = ORTModelForAudioClassification.from_pretrained(
    model_name,
    export=True,                      # force export
    provider="CPUExecutionProvider"
)

# 3) Save to disk
save_dir = "/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2/onnx/wav2vec2-base-audioreplay-onnx"
ort_model.save_pretrained(save_dir)
feature_extractor.save_pretrained(save_dir)

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/mnt/media/miniconda3/envs/telephone_audio_generator_py311/lib/python3.11/site-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:839: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if attn_output.size() != (bsz, self.num_heads, tgt_len, self.head_dim):


['/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2/onnx/wav2vec2-base-audioreplay-onnx/preprocessor_config.json']

### Load the ONNX model and run an inference test

In [ ]:
from transformers import Wav2Vec2FeatureExtractor
from optimum.onnxruntime import ORTModelForAudioClassification
import librosa
import torch

# 1) Directory where you saved model.onnx + config.json + preprocessor_config.json
save_dir = "/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2/onnx/wav2vec2-base-audioreplay-onnx"

# 2) Path to the audio file you want to classify
audio_path = "/mnt/media/fair/audio/replay_attacks/datasets/ASVSpoof2017/ASVspoof2017_V2_train/ASVspoof2017_V2_train/T_1000001.wav"

# --- LOAD ONNX MODEL AND FEATURE EXTRACTOR ---
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(save_dir)
ort_model = ORTModelForAudioClassification.from_pretrained(save_dir)

# --- AUDIO PREPROCESSING ---
audio, sr = librosa.load(audio_path, sr=feature_extractor.sampling_rate)
inputs = feature_extractor(
    audio,
    sampling_rate=sr,
    return_tensors="pt",      # ONNX Runtime in Optimum accepts PyTorch tensors
    padding=False,
    truncation=False
)

# --- ONNX INFERENCE ---
with torch.no_grad():
    outputs = ort_model(**inputs)
    logits = outputs.logits               # shape (1, num_labels)
    probs = torch.softmax(logits, dim=-1) # convert logits to probabilities
    pred_id = torch.argmax(probs, dim=-1).item()

# --- DISPLAY RESULTS ---
print(f"Logits:         {logits.cpu().numpy()}")
print(f"Probabilities:  {probs.cpu().numpy()}")
print(f"Predicted class: {pred_id}")


Logits:         [[-0.00235653  0.06352573]]
Probabilidades: [[0.48353538 0.5164646 ]]
Clase predicha: 1


## Load a pretrained SpeechBrain model using Optimum / HuggingFace

In [ ]:
import torch
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

# 1) Base directory where your checkpoint folders are located
ckpt_dir = "/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2/26/save/CKPT+2025-07-20+18-28-54+00"

# 2) Load the two main components from your SpeechBrain model:
#    - wav2vec2.ckpt: encoder weights
#    - output_mlp.ckpt: classification head weights
wav2vec_ckpt    = torch.load(f"{ckpt_dir}/wav2vec2.ckpt",    map_location="cpu")
output_mlp_ckpt = torch.load(f"{ckpt_dir}/output_mlp.ckpt",  map_location="cpu")

# 3) Instantiate exactly the same architecture in HF that you trained in SpeechBrain
hf_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=2,
    problem_type="single_label_classification"
)

# 4) Extract the state_dicts from SpeechBrain
sb_w2v_dict  = wav2vec_ckpt.get("state_dict", wav2vec_ckpt)
sb_out_dict  = output_mlp_ckpt.get("state_dict", output_mlp_ckpt)

# 5) Combine them into a single dict. Adjust prefixes if your keys differ:
combined_dict = {}
combined_dict.update(sb_w2v_dict)
combined_dict.update(sb_out_dict)

fixed_dict = {}
for k, v in combined_dict.items():
    if k.startswith("model."):
        new_key = k.replace("model.", "wav2vec2.", 1)
    else:
        new_key = k
    fixed_dict[new_key] = v

# 6) Load the weights into the HF model (strict=False to ignore unmatched keys)
missing, unexpected = hf_model.load_state_dict(fixed_dict, strict=False)
print("MISSING keys (not loaded):", missing)
print("UNEXPECTED keys (extra in checkpoint):", unexpected)


/mnt/media/miniconda3/envs/telephone_audio_generator_py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/media/miniconda3/envs/telephone_audio_generator_py311/lib/python3.11/site-packages/transformers/configuration_utils.py:312: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-

MISSING keys (no cargadas): ['projector.weight', 'projector.bias', 'classifier.weight', 'classifier.bias']
UNEXPECTED keys (sobrantes): ['0.w.weight']


In [ ]:
import torch
import torch.nn as nn
from transformers import Wav2Vec2ForSequenceClassification

# 1) Load the original HF model and redefine the head from 768→2
hf_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=2,
    problem_type="single_label_classification"
)
hf_model.projector = nn.Identity()
hf_model.classifier = nn.Linear(hf_model.config.hidden_size,
                                hf_model.config.num_labels)

ckpt_dir = "/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2/26/save/CKPT+2025-07-20+18-28-54+00"

# 2) Load your SpeechBrain state_dicts
wav2vec_ckpt = torch.load(f"{ckpt_dir}/wav2vec2.ckpt", map_location="cpu").get("state_dict")
output_mlp_ckpt = torch.load(f"{ckpt_dir}/output_mlp.ckpt", map_location="cpu").get("state_dict")
sb_w2v_dict = wav2vec_ckpt.get("state_dict", wav2vec_ckpt)
sb_out_dict = output_mlp_ckpt.get("state_dict", output_mlp_ckpt)

# 3) Build the fixed_dict
fixed_dict = {}

# 3a) Encoder: 'model.xxx' → 'wav2vec2.xxx'
for k, v in sb_w2v_dict.items():
    fixed_dict[k.replace("model.", "wav2vec2.", 1)] = v

# 3b) Classification head: 0.w.weight → classifier.weight
fixed_dict["classifier.weight"] = sb_out_dict["0.w.weight"]
# Initialize bias to zeros if "0.b.bias" is not found
if "0.b.bias" in sb_out_dict:
    fixed_dict["classifier.bias"] = sb_out_dict["0.b.bias"]
else:
    fixed_dict["classifier.bias"] = torch.zeros(hf_model.config.num_labels)

# 4) Load everything at once into the full model
missing, unexpected = hf_model.load_state_dict(fixed_dict, strict=False)
print("Missing:", missing)
print("Unexpected:", unexpected)


Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AttributeError: 'NoneType' object has no attribute 'get'